# TfL Bikes: Exploring the Relationship Between Bikes Hired and Various Variables

In this notebook, we'll use data from [TfL (Transport for London)](http://www.tfl.gov.uk) to analyse usage of the London Bike Sharing scheme.

We'll cover:
1. Loading and cleaning the data
2. Summary statistics
3. Exploratory Data Analysis (EDA) with visualisations
4. Regression model building
5. Model diagnostics and comparison

## 1. Import Libraries

Before we can do anything, we need to import the Python libraries (packages) we'll use.


In [1]:
# pandas is the core data manipulation library in Python.
# It gives us the DataFrame object 
import pandas as pd

# numpy provides numerical operations (like calculating means, arrays, etc.)
import numpy as np

# matplotlib is the foundational plotting library in Python.
# We import pyplot (the main interface) and alias it as 'plt'.
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# seaborn is a higher-level plotting library built on top of matplotlib.
# It's the Python equivalent of ggplot2 -- great for statistical visualisations.
import seaborn as sns

# scipy.stats gives us statistical tests like the t-test
from scipy import stats

# statsmodels is used for regression modelling (like lm() in R)
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# warnings is a built-in Python module — we use it to suppress noisy warnings
import warnings
warnings.filterwarnings('ignore')

# This line makes matplotlib plots render inline inside the notebook
%matplotlib inline

# Set a consistent visual style for all seaborn plots (like theme_bw() in ggplot2)
sns.set_theme(style='whitegrid', palette='muted')

# Make figures a bit larger by default so they're easier to read
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
## 2. Load the Data

We read the CSV file into a pandas DataFrame using `pd.read_csv()`.


In [17]:
# Load the CSV and immediately start a method chain to clean/enrich it.

bike = (
    # Step 1: Read the CSV file into a DataFrame
    # Change the path below if your file is in a different location
    pd.read_csv('https://raw.githubusercontent.com/kostis-christodoulou/data_analytics_executives/main/data/london_bikes.csv')

    # Step 2: Parse the 'date' column as an actual date type
    # By default, pandas reads dates as plain strings — we need to convert them.
    .assign(date = lambda df: pd.to_datetime(df['date']))

    # Step 3: Create a 'weekend' boolean column
    # .dt.day_name() returns the day name (e.g. 'Monday', 'Saturday').
    # .isin() checks if the value is in our list 
    .assign(weekend = lambda df: df['date'].dt.day_name().isin(['Saturday', 'Sunday']))
)

# Preview the first few rows — like head() in R
print(f"Dataset shape: {bike.shape[0]} rows × {bike.shape[1]} columns")
bike.head()






Dataset shape: 5846 rows × 33 columns


,date,bikes_hired,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon,weekend
0,2010-07-30 00:00:00+00:00,6897,21.5,10.7,16.8,21.5,10.7,16.8,10.3,68.1,...,23.9,9,NaN,2010-07-30T05:17:39Z,2010-07-30T20:50:39Z,0.65,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,False
1,2010-07-31 00:00:00+00:00,5564,23.0,15.0,18.9,23.0,15.0,18.9,14.3,76.3,...,19.6,7,NaN,2010-07-31T05:19:09Z,2010-07-31T20:49:03Z,0.68,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,True
2,2010-08-01 00:00:00+00:00,4303,21.0,14.0,17.4,21.0,14.0,17.4,11.5,69.1,...,22.3,8,NaN,2010-08-01T05:20:39Z,2010-08-01T20:47:25Z,0.71,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,True
3,2010-08-02 00:00:00+00:00,6642,20.4,14.3,17.1,20.4,14.3,17.1,11.6,71.1,...,23.0,7,NaN,2010-08-02T05:22:11Z,2010-08-02T20:45:45Z,0.75,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,False
4,2010-08-03 00:00:00+00:00,7966,21.6,11.6,17.2,21.6,11.6,17.2,11.4,71.4,...,24.7,8,NaN,2010-08-03T05:23:42Z,2010-08-03T20:44:03Z,0.75,"Rain, Partially cloudy",Becoming cloudy in the afternoon with rain cle...,rain,False


---
## 3. Cleaning & Enriching the Data

Now we add several new date-based columns and a `season_name` column.
In Python we use pandas' built-in `.dt` accessor and `np.select()`.

- `.dt.year` → `year(date)` in R
- `.dt.month` → `month(date)` in R
- `.dt.month_name()` → `month(date, label=TRUE)` in R
- `.dt.day_name()` → `wday(date, label=TRUE)` in R
- `np.select()` → `case_when()` in R
- `pd.Categorical()` → `factor(..., levels=...)` in R

In [3]:
# We continue the data enrichment using .assign(), chaining multiple new columns.
# Each lambda function receives the current state of the DataFrame (df)
# and returns a new column — just like mutate() in dplyr.

bike = (
    bike

    # Extract the year as a number (e.g. 2015, 2016)
    .assign(year = lambda df: df['date'].dt.year)

    # Extract the month as a number (1 = January, 12 = December)
    .assign(month = lambda df: df['date'].dt.month)

    # Extract the abbreviated month name (e.g. 'Jan', 'Feb', ...)
    # strftime('%b') formats the date to a short month name string
    .assign(month_name = lambda df: df['date'].dt.strftime('%b'))

    # Extract the abbreviated day of the week (e.g. 'Mon', 'Tue', ...)
    .assign(day_of_week = lambda df: df['date'].dt.strftime('%a'))

    # Create a season_name column using np.select(), which is like case_when() in R.
    # We define a list of conditions and a corresponding list of choices.
    # np.select() goes through conditions in order and picks the first match.
    .assign(season_name = lambda df: np.select(
        condlist=[
            df['month_name'].isin(['Dec', 'Jan', 'Feb']),  # Winter
            df['month_name'].isin(['Mar', 'Apr', 'May']),  # Spring
            df['month_name'].isin(['Jun', 'Jul', 'Aug']),  # Summer
            df['month_name'].isin(['Sep', 'Oct', 'Nov']),  # Autumn
        ],
        choicelist=['Winter', 'Spring', 'Summer', 'Autumn'],
        default='Unknown'  # fallback if no condition matches (shouldn't happen)
    ))

    # Convert season_name to an ordered Categorical — like factor(..., levels=...) in R.
    # This ensures seasons appear in the right order in plots and tables,
    # rather than alphabetically.
    .assign(season_name = lambda df: pd.Categorical(
        df['season_name'],
        categories=['Winter', 'Spring', 'Summer', 'Autumn'],
        ordered=True
    ))

    # Similarly, order the months correctly (Jan, Feb, ..., Dec)
    .assign(month_name = lambda df: pd.Categorical(
        df['month_name'],
        categories=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
        ordered=True
    ))

    # Order days of the week starting from Monday (like R's wday with week_start=1)
    .assign(day_of_week = lambda df: pd.Categorical(
        df['day_of_week'],
        categories=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
        ordered=True
    ))
)

print("✅ Data cleaned and enriched!")
print(f"\nColumn names: {list(bike.columns)}")
bike.head()

✅ Data cleaned and enriched!

Column names: ['date', 'bikes_hired', 'tempmax', 'tempmin', 'temp', 'feelslikemax', 'feelslikemin', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob', 'precipcover', 'preciptype', 'snow', 'snowdepth', 'windgust', 'windspeed', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'sunrise', 'sunset', 'moonphase', 'conditions', 'description', 'icon', 'weekend', 'year', 'month', 'month_name', 'day_of_week', 'season_name']


,date,bikes_hired,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,moonphase,conditions,description,icon,weekend,year,month,month_name,day_of_week,season_name
0,2010-07-30 00:00:00+00:00,6897,21.5,10.7,16.8,21.5,10.7,16.8,10.3,68.1,...,0.65,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,False,2010,7,Jul,Fri,Summer
1,2010-07-31 00:00:00+00:00,5564,23.0,15.0,18.9,23.0,15.0,18.9,14.3,76.3,...,0.68,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,True,2010,7,Jul,Sat,Summer
2,2010-08-01 00:00:00+00:00,4303,21.0,14.0,17.4,21.0,14.0,17.4,11.5,69.1,...,0.71,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,True,2010,8,Aug,Sun,Summer
3,2010-08-02 00:00:00+00:00,6642,20.4,14.3,17.1,20.4,14.3,17.1,11.6,71.1,...,0.75,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,False,2010,8,Aug,Mon,Summer
4,2010-08-03 00:00:00+00:00,7966,21.6,11.6,17.2,21.6,11.6,17.2,11.4,71.4,...,0.75,"Rain, Partially cloudy",Becoming cloudy in the afternoon with rain cle...,rain,False,2010,8,Aug,Tue,Summer


### 3.1 Examine the structure of the data

In Python, we use a combination of `.info()` and `.describe()` to get quick information on the dataset

In [4]:
# .info() shows us column names, non-null counts, and data types.
# This is like str() in R — it tells you the structure of your data.
print("=== DataFrame Structure ===")
bike.info()

=== DataFrame Structure ===
<class 'pandas.DataFrame'>
RangeIndex: 5846 entries, 0 to 5845
Data columns (total 38 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   date              5846 non-null   datetime64[us, UTC]
 1   bikes_hired       5846 non-null   int64              
 2   tempmax           5846 non-null   float64            
 3   tempmin           5846 non-null   float64            
 4   temp              5846 non-null   float64            
 5   feelslikemax      5846 non-null   float64            
 6   feelslikemin      5846 non-null   float64            
 7   feelslike         5846 non-null   float64            
 8   dew               5846 non-null   float64            
 9   humidity          5846 non-null   float64            
 10  precip            5846 non-null   float64            
 11  precipprob        5846 non-null   int64              
 12  precipcover       5846 non-null   float64    

In [ ]:
# .describe() computes summary statistics for all numeric columns:
# count, mean, std (standard deviation), min, quartiles, and max.

# .T transposes the table (rows become columns) so it's easier to read.
bike.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
bikes_hired,5846.0,26183.16,9237.99,0.0,19952.00,26056.50,32106.75,73094.00
tempmax,5846.0,14.91,6.45,-2.6,10.10,14.60,19.60,36.10
tempmin,5846.0,7.95,5.09,-11.1,4.10,8.10,11.80,22.10
temp,5846.0,11.37,5.55,-5.9,7.20,11.30,15.60,28.30
feelslikemax,5846.0,14.28,7.30,-7.2,10.10,14.60,19.60,38.20
feelslikemin,5846.0,6.35,6.40,-14.6,1.30,6.10,11.80,22.10
feelslike,5846.0,10.27,6.70,-9.1,5.00,10.90,15.60,28.90
dew,5846.0,7.40,4.68,-8.0,4.00,7.60,11.00,19.00
humidity,5846.0,78.57,10.68,39.6,71.20,79.70,86.80,99.90
precip,5846.0,1.57,4.59,0.0,0.00,0.16,1.19,168.20


---
## 4. Summary Statistics

In Python, we use pandas' `.agg()` method to compute multiple statistics at once,
and `.groupby()` to split by a categorical variable 

In [12]:
# Summary statistics GROUPED by year, or think of it as bikes_hired ~ year

# .groupby() splits the DataFrame by the given column
# .agg() then applies our chosen statistics within each group
# .round(1) rounds all values to 1 decimal place

def grouped_stats(df, group_col, value_col='bikes_hired'):
    """Helper function: compute summary stats grouped by a column.
    
    Arguments:
        df        -- the DataFrame to summarise
        group_col -- the column to group by (like the X in favstats(Y ~ X))
        value_col -- the variable to summarise (default: 'bikes_hired')
    """
    return (
        df
        .groupby(group_col, observed=True)[value_col]  # group and select the target column
        .agg(
            min='min',
            Q1=lambda x: x.quantile(0.25),
            median='median',
            mean='mean',
            Q3=lambda x: x.quantile(0.75),
            max='max',
            std='std',
            n='count'
        )
        .round(1)
    )

print("=== Summary by Year ===")
grouped_stats(bike, 'year')

=== Summary by Year ===


,min,Q1,median,mean,Q3,max,std,n
year,,,,,,,,
2010,2764,9297.0,14010.0,14069.8,18677.5,27964,5616.9,155
2011,4555,16260.0,20264.0,19568.4,23708.0,29417,5497.5,365
2012,3531,19282.0,26178.5,26009.0,32532.8,47102,9429.4,366
2013,3728,17555.0,22021.0,22042.4,27371.0,35580,7276.5,365
2014,4327,20532.0,27676.0,27462.7,34437.0,49025,9065.1,365
2015,5779,22056.0,26618.0,27046.1,32857.0,73094,8547.9,365
2016,4894,22402.0,27881.5,28152.0,35130.0,46625,8851.0,366
2017,5143,24064.0,29490.0,28619.3,34459.0,46035,8376.6,365
2018,5859,21759.0,29190.0,28952.2,37677.0,46084,10174.3,365


In [ ]:
# Summary by day of week
# Equivalent to: favstats(bikes_hired ~ day_of_week, data = bike)
print("=== Summary by Day of Week ===")
grouped_stats(bike, 'day_of_week')

In [ ]:
# Summary by month name
# Equivalent to: favstats(bikes_hired ~ month_name, data = bike)
print("=== Summary by Month ===")
grouped_stats(bike, 'month_name')

In [ ]:
# Summary by season
# Equivalent to: favstats(bikes_hired ~ season_name, data = bike)
print("=== Summary by Season ===")
grouped_stats(bike, 'season_name')

---
## 5. Exploratory Data Analysis (EDA)

Summary statistics alone don't tell the whole story — visualisations help reveal trends,
patterns, seasonality, and outliers that numbers alone might hide.

### 5.1 Time Series Scatter Plot

First, we filter our data to start from 2014 (to match the R notebook),
then create a scatter plot of `bikes_hired` over time.

The equivalent in R was:
```r
bike %>% filter(date >= ymd("2014-01-01")) %>% ggplot() + aes(x=date, y=bikes_hired) + geom_point(alpha=0.4)
```

In [ ]:
# Filter to 2014 onwards — like dplyr's filter(date >= ymd("2014-01-01"))
# The result is stored in bike_2014 for reuse in subsequent plots.
bike_2014 = (
    bike
    .query("date >= '2014-01-01'")  # .query() is like dplyr's filter()
    .reset_index(drop=True)          # reset row numbers after filtering
)

# Create a scatter plot of bikes hired vs. date
fig, ax = plt.subplots(figsize=(12, 5))

# sns.scatterplot is like geom_point() in ggplot2.
# alpha controls transparency (0 = fully transparent, 1 = fully opaque)
# — useful when points overlap (overplotting)
sns.scatterplot(
    data=bike_2014,
    x='date',
    y='bikes_hired',
    alpha=0.4,   # 40% opacity — same as geom_point(alpha=0.4)
    ax=ax        # draw on our specific axes object
)

ax.set_title('Daily Bike Rentals Over Time (2014 onwards)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Bikes Hired')
plt.tight_layout()
plt.show()

### 5.2 Histograms

Histograms show the distribution of a single variable.
The x-axis represents value ranges (bins) and the y-axis shows how many observations fall in each bin.

In [ ]:
# Simple histogram of bikes_hired — like geom_histogram() in ggplot2
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(
    data=bike_2014,
    x='bikes_hired',
    bins=30,       # how many bars to use (more bins = finer detail)
    ax=ax
)

ax.set_title('Distribution of Daily Bikes Hired', fontsize=14)
ax.set_xlabel('Bikes Hired')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Histograms faceted by season — like facet_wrap(~season_name) in ggplot2
# In seaborn, we use FacetGrid to create small multiples (one plot per group).

# FacetGrid creates a grid of subplots, one for each unique value of 'col'
g = sns.FacetGrid(
    data=bike_2014,
    col='season_name',   # one column (subplot) per season
    col_wrap=2,          # wrap into 2 columns (so 2x2 grid for 4 seasons)
    height=4,
    aspect=1.4,
    sharey=False         # let each subplot have its own y-axis scale
)

# .map() applies the same plot function to each facet
g.map(sns.histplot, 'bikes_hired', bins=25)

g.set_axis_labels('Bikes Hired', 'Count')
g.set_titles(col_template='{col_name}')  # season name as subplot title
g.figure.suptitle('Distribution of Bikes Hired by Season', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms faceted by month — like facet_wrap(~month_name, nrow=4) in ggplot2
# col_order ensures months appear in calendar order (Jan → Dec)

g = sns.FacetGrid(
    data=bike_2014,
    col='month_name',
    col_wrap=4,   # 4 columns = 3 rows for 12 months
    height=3,
    aspect=1.2,
    sharey=False,
    col_order=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
g.map(sns.histplot, 'bikes_hired', bins=20)
g.set_axis_labels('Bikes Hired', 'Count')
g.set_titles(col_template='{col_name}')
g.figure.suptitle('Distribution of Bikes Hired by Month', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Density Plots

A density plot is a smoothed version of a histogram.
It's useful for comparing the shape of distributions between groups.
In R this was `geom_density()` in ggplot2.

In [ ]:
# Simple density plot — like geom_density() in ggplot2
fig, ax = plt.subplots(figsize=(10, 5))

sns.kdeplot(   # KDE = Kernel Density Estimate — the smooth curve
    data=bike_2014,
    x='bikes_hired',
    fill=True,   # shade under the curve (like geom_density with fill)
    ax=ax
)

ax.set_title('Density of Daily Bikes Hired', fontsize=14)
ax.set_xlabel('Bikes Hired')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
# Overlapping density plots by season — like geom_density(aes(fill=season_name), alpha=0.3)
# Overlaying multiple seasons on the same plot makes it easy to compare their distributions.

fig, ax = plt.subplots(figsize=(10, 5))

# hue='season_name' tells seaborn to draw one curve per season, using different colours
sns.kdeplot(
    data=bike_2014,
    x='bikes_hired',
    hue='season_name',  # colour by season (like aes(fill=season_name) in ggplot2)
    fill=True,
    alpha=0.3,          # semi-transparent so overlapping curves are visible
    ax=ax
)

ax.set_title('Density of Bikes Hired by Season', fontsize=14)
ax.set_xlabel('Bikes Hired')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
# Faceted density plots by month, coloured by season
# Like: geom_density(aes(fill=season_name)) + facet_wrap(~month_name, nrow=4)

# First, create a colour mapping: which season does each month belong to?
season_colours = {
    'Winter': '#5B9BD5',
    'Spring': '#70AD47',
    'Summer': '#FFC000',
    'Autumn': '#ED7D31'
}

# Map each row's month_name to its season colour (used for fill in the plot)
month_to_season = (
    bike_2014
    .drop_duplicates('month_name')[['month_name', 'season_name']]
    .set_index('month_name')['season_name']
    .to_dict()
)

months_ordered = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey=False)
axes = axes.flatten()  # turn 2D array of axes into a 1D list for easy indexing

for i, month in enumerate(months_ordered):
    # Filter data to just this month
    month_data = bike_2014.query("month_name == @month")
    season = month_to_season.get(month, 'Winter')
    colour = season_colours[season]

    # Draw histogram on the corresponding subplot
    sns.histplot(
        data=month_data,
        x='bikes_hired',
        color=colour,
        alpha=0.6,
        bins=15,
        ax=axes[i]
    )
    axes[i].set_title(month, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count' if i % 4 == 0 else '')  # only show y-label on left column

fig.suptitle('Distribution of Bikes Hired by Month (coloured by Season)', fontsize=14)
plt.tight_layout()
plt.show()

### 5.4 Box Plots

Box plots show the median, quartiles, and outliers of a distribution.
They are great for comparing groups side-by-side.
In R this was `geom_boxplot()` in ggplot2.

In [ ]:
# Boxplot of bikes_hired by month
# Like: aes(x=month_name, y=bikes_hired) + geom_boxplot()

fig, ax = plt.subplots(figsize=(12, 5))

sns.boxplot(
    data=bike_2014,
    x='month_name',   # x-axis: the grouping variable
    y='bikes_hired',  # y-axis: the numeric variable we're summarising
    order=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],  # enforce calendar order
    ax=ax
)

ax.set_title('Bikes Hired by Month (Box Plot)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Bikes Hired')
plt.tight_layout()
plt.show()

### 5.5 Scatter Plots: Bikes vs. Weather Variables

Now let's look at how weather variables relate to bike rentals.
We'll use regression trend lines (like `geom_smooth(method='lm')` in ggplot2)
to summarise the relationship.

In [ ]:
# Scatter plot: bikes_hired vs. temperature, coloured by season
# Like: aes(x=temp, y=bikes_hired, colour=season_name) + geom_point() + geom_smooth(method='lm')

fig, ax = plt.subplots(figsize=(10, 6))

# sns.lmplot() or sns.regplot() both draw a scatter + regression line.
# Using a loop here so we can draw one line per season (like colour=season_name in ggplot2)
palette = {'Winter': '#5B9BD5', 'Spring': '#70AD47', 'Summer': '#FFC000', 'Autumn': '#ED7D31'}

for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    season_df = bike_2014.query("season_name == @season")
    sns.regplot(
        data=season_df,
        x='temp',
        y='bikes_hired',
        scatter_kws={'alpha': 0.1, 'color': palette[season]},  # points: transparent
        line_kws={'color': palette[season], 'linewidth': 2},    # line: solid
        label=season,
        ax=ax
    )

ax.set_title('Bikes Hired vs. Temperature (by Season)', fontsize=14)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Bikes Hired')
ax.legend(title='Season')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: bikes_hired vs. humidity, coloured by season
# Same structure as above but with 'humidity' on the x-axis

fig, ax = plt.subplots(figsize=(10, 6))

for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    season_df = bike_2014.query("season_name == @season")
    sns.regplot(
        data=season_df,
        x='humidity',
        y='bikes_hired',
        scatter_kws={'alpha': 0.1, 'color': palette[season]},
        line_kws={'color': palette[season], 'linewidth': 2},
        label=season,
        ax=ax
    )

ax.set_title('Bikes Hired vs. Humidity (by Season)', fontsize=14)
ax.set_xlabel('Humidity (%)')
ax.set_ylabel('Bikes Hired')
ax.legend(title='Season')
plt.tight_layout()
plt.show()

---
## 6. Model Building

### 6.1 Correlation — Scatterplot Matrix

Before fitting regression models, it's useful to look at how all the numeric variables
relate to each other. In R this was done with `GGally::ggpairs()`.
In Python, seaborn's `pairplot()` does the same thing.

In [ ]:
# Select the variables we care about
# Like: bike %>% select(cloudcover, humidity, precip, temp, bikes_hired) %>% GGally::ggpairs()

pair_cols = ['cloudcover', 'humidity', 'precip', 'temp', 'bikes_hired']

pair_data = (
    bike_2014
    [pair_cols]        # select only those columns — like dplyr's select()
    .dropna()          # drop any rows with missing values
)

# pairplot creates a grid:
# - diagonal: distribution of each variable
# - off-diagonal: scatter plots of each variable pair
g = sns.pairplot(
    pair_data,
    diag_kind='kde',      # kernel density on diagonal (like the density plots above)
    plot_kws={'alpha': 0.2}  # make points semi-transparent
)
g.figure.suptitle('Scatterplot Matrix of Key Variables', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Weekend vs. Weekday: Is There a Difference?

We'll use an independent samples t-test to check whether weekends have
statistically different bike rentals compared to weekdays.
In R this was `t.test(bikes_hired ~ weekend, data = bike)`.

In [ ]:
# Split bikes_hired into two groups: weekdays and weekends
weekend_group  = bike.query("weekend == True")['bikes_hired']
weekday_group  = bike.query("weekend == False")['bikes_hired']

# Run a two-sample t-test
# H0: the two means are equal
# H1: the means are different (two-sided test)
t_stat, p_value = stats.ttest_ind(weekend_group, weekday_group)

print("=== Independent Samples T-Test: Weekend vs. Weekday ===")
print(f"  Weekend mean:  {weekend_group.mean():,.0f} bikes/day")
print(f"  Weekday mean:  {weekday_group.mean():,.0f} bikes/day")
print(f"  T-statistic:   {t_stat:.3f}")
print(f"  P-value:       {p_value:.4f}")
print()
if p_value < 0.05:
    print("  ✅ The difference IS statistically significant (p < 0.05).")
else:
    print("  ❌ The difference is NOT statistically significant (p >= 0.05).")

---
## 7. Regression Models

We'll use `statsmodels` for regression — the Python equivalent of `lm()` in R.

The `smf.ols()` function uses R-style **formula syntax** (e.g. `'bikes_hired ~ temp'`),
which makes the transition from R very natural.

**Helper function** to avoid repeating code: we create a function that fits a model,
prints the summary, and plots actual vs. fitted values — since we do this for every model.

In [ ]:
def fit_and_plot(formula, data, model_name="Model"):
    """
    Fit an OLS regression model and visualise actual vs. fitted values.
    
    This function does three things:
      1. Fits the model using the formula (like lm() in R)
      2. Prints the model summary (like msummary() or summary() in R)
      3. Plots actual (black) vs. fitted (red) values over time
         — like broom::augment() + ggplot in R
    
    Arguments:
        formula    -- model formula as a string, e.g. 'bikes_hired ~ temp + weekend'
        data       -- the DataFrame to use
        model_name -- a label for the plot title
    
    Returns:
        The fitted model object (so you can inspect it further if needed)
    """
    # Fit the OLS model — like lm(formula, data=bike) in R
    model = smf.ols(formula=formula, data=data).fit()

    # Print the summary table — like summary(model) or msummary(model) in R
    print(model.summary())
    
    # ---- Plot actual vs. fitted ----
    # model.fittedvalues is like .fitted from broom::augment() in R
    fig, ax = plt.subplots(figsize=(12, 5))

    # Day number (1, 2, 3...) on the x-axis — like mutate(day = row_number()) in R
    days = range(len(data))

    # Actual values as filled black dots
    ax.scatter(days, data['bikes_hired'], alpha=0.2, color='black', s=10, label='Actual')

    # Fitted (predicted) values as open red circles
    ax.scatter(days, model.fittedvalues, alpha=0.2, color='red',
               s=10, marker='o', facecolors='none', label='Fitted')

    ax.set_title(f'{model_name}: Actual vs. Fitted Values', fontsize=13)
    ax.set_xlabel('Day (index)')
    ax.set_ylabel('Bikes Hired')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return model

print("✅ Helper function defined — ready to fit models!")

### Model 0: Naive Baseline — Predict with the Mean

The simplest possible model is to always predict the average number of bikes hired.
In regression notation: `bikes_hired ~ 1` (intercept only, no predictors).
The intercept will equal the mean of `bikes_hired`.
This gives us a baseline to compare all other models against.

In [ ]:
# Model 0: intercept-only model — like lm(bikes_hired ~ 1, data = bike) in R
# The formula '1' means: just fit a constant (the mean)
model0 = fit_and_plot(
    formula='bikes_hired ~ 1',
    data=bike,
    model_name='Model 0 (Mean-only Baseline)'
)

### Model 1: Temperature as a Predictor

Does minimum temperature help predict bike rentals?
In R: `lm(bikes_hired ~ tempmin, data = bike)`

In [ ]:
# Model 1: one predictor — minimum temperature
model1 = fit_and_plot(
    formula='bikes_hired ~ tempmin',
    data=bike,
    model_name='Model 1 (tempmin)'
)

# Questions to reflect on:
# - Is the coefficient on tempmin statistically significant? Look at the p-value.
# - What is the R-squared? It tells you what % of variance in bikes_hired is explained by tempmin.

### Model 2: Feels-like Temperature + Weekend + Visibility

In [ ]:
# Model 2: multiple predictors
# 'weekend' is a boolean (True/False) — statsmodels treats it as a dummy variable automatically
# Like: lm(bikes_hired ~ feelslikemax + weekend + visibility, data = bike)
model2 = fit_and_plot(
    formula='bikes_hired ~ feelslikemax + weekend + visibility',
    data=bike,
    model_name='Model 2 (feelslikemax + weekend + visibility)'
)

# Questions to reflect on:
# - What is the meaning of the weekend[T.True] coefficient?
#   It represents the average CHANGE in bikes hired on weekends vs. weekdays,
#   holding all other variables constant.
# - Did adding these predictors improve R-squared compared to Model 1?

### Model 3: Feels-like Temperature + Day of Week

`wday` is a categorical variable with 7 levels (Mon–Sun).
statsmodels will automatically create 6 dummy variables (one level is dropped as the reference).
The coefficient for, say, `wday[T.Mon]` means: "compared to the reference day (Friday),
how many more/fewer bikes were hired on Mondays, on average?"

In [ ]:
# Model 3: feelslikemax + day of week (categorical)
# C() tells statsmodels to treat a variable as categorical (creates dummies)
# Like: lm(bikes_hired ~ feelslikemax + wday, data = bike)
model3 = fit_and_plot(
    formula='bikes_hired ~ feelslikemax + C(wday)',
    data=bike,
    model_name='Model 3 (feelslikemax + day of week)'
)

### Model 4: Feels-like Temp + Day of Week + Humidity + Month

In [ ]:
# Model 4: adding humidity and month as additional predictors
# C(month) treats the numeric month as a categorical variable (12 dummies)
# Like: lm(bikes_hired ~ feelslikemax + wday + humidity + factor(month), data = bike)
model4 = fit_and_plot(
    formula='bikes_hired ~ feelslikemax + C(wday) + humidity + C(month)',
    data=bike,
    model_name='Model 4 (+ humidity + month)'
)

### Model 5: Adding Precipitation and Wind Speed

In [ ]:
# Model 5: further adding precipitation and windspeed
model5 = fit_and_plot(
    formula='bikes_hired ~ feelslikemax + C(wday) + humidity + C(month) + precip + windspeed',
    data=bike,
    model_name='Model 5 (+ precip + windspeed)'
)

### Model 6 & 7: Temperature + Day of Week + Humidity + Month + Cloud Cover + Moon Phase

In [ ]:
# Model 6: swap feelslikemax for temp, add cloudcover and moonphase
# moonphase is a continuous variable (0 = new moon, 0.5 = full moon, 1 = new moon again)
model6 = fit_and_plot(
    formula='bikes_hired ~ temp + C(wday) + humidity + C(month) + cloudcover + moonphase',
    data=bike,
    model_name='Model 6 (temp + cloud + moon)'
)

In [ ]:
# Model 7 is the same formula as model 6 in the original R notebook
# (You can modify this to try your own variable combinations!)
model7 = model6  # same model — placeholder for your own experimentation

---
## 8. Best Model: Actual vs. Fitted Visualisation

In the R notebook, `model3` was suggested as a 'best' model.
Here we reproduce the actual vs. fitted plot more clearly.

In [ ]:
# Build a DataFrame with actual values, fitted values, and residuals
# Like: broom::augment(model3) %>% mutate(day = row_number()) in R

# We use .assign() to add columns to a copy of the bike data
best_model_df = (
    bike
    .assign(day = lambda df: range(len(df)))      # day index (like row_number())
    .assign(fitted = model3.fittedvalues.values)  # predicted values from model3
    .assign(residual = lambda df: df['bikes_hired'] - df['fitted'])  # actual - fitted
)

# Plot actual (black) and fitted (red) on the same chart
fig, ax = plt.subplots(figsize=(13, 5))

ax.scatter(best_model_df['day'], best_model_df['bikes_hired'],
           alpha=0.2, color='black', s=10, label='Actual')

ax.scatter(best_model_df['day'], best_model_df['fitted'],
           alpha=0.3, color='red', s=10, facecolors='none',
           marker='o', label='Fitted (Model 3)')

ax.set_title('Model 3: Actual vs. Fitted Bike Rentals', fontsize=14)
ax.set_xlabel('Day Index')
ax.set_ylabel('Bikes Hired')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Diagnostics: Variance Inflation Factor (VIF)

When we add many predictors to a model, some may be highly correlated with each other.
This is called **multicollinearity**, and it can make our coefficient estimates unreliable.

The **Variance Inflation Factor (VIF)** quantifies how much each predictor's coefficient
variance is inflated due to collinearity with other predictors.

- VIF = 1: no collinearity
- VIF 1–5: acceptable
- VIF > 10: problematic — consider removing or combining the variable

In R this was `car::vif(model_x)`. In Python we use `statsmodels`.

In [ ]:
def compute_vif(model, data):
    """
    Compute the Variance Inflation Factor (VIF) for each predictor in a fitted model.
    
    High VIF (>10) indicates multicollinearity: the predictor is nearly redundant
    because it can be predicted from the other predictors.
    
    Arguments:
        model -- a fitted statsmodels OLS model
        data  -- the DataFrame used to fit the model
    
    Returns:
        A DataFrame showing VIF for each predictor
    """
    # Build the design matrix X (the matrix of predictor values)
    # This is what statsmodels uses internally — adding a constant column for the intercept
    X = sm.add_constant(model.model.exog)

    # variance_inflation_factor() requires the design matrix and the column index
    vif_data = pd.DataFrame({
        'Variable': model.model.exog_names,
        'VIF': [variance_inflation_factor(X, i+1) for i in range(len(model.model.exog_names))]
    }).sort_values('VIF', ascending=False)

    return vif_data

# Compute VIF for Model 2
print("=== VIF for Model 2 ===")
print(compute_vif(model2, bike).to_string(index=False))
print()

# Compute VIF for Model 3
print("=== VIF for Model 3 ===")
print(compute_vif(model3, bike).to_string(index=False))
print()

# Compute VIF for Model 4
print("=== VIF for Model 4 ===")
print(compute_vif(model4, bike).to_string(index=False))

---
## 10. Model Comparison Table

In R, `huxtable::huxreg()` creates a side-by-side table of multiple models.
Here we build a similar comparison table using pandas, showing:
- Number of observations
- R-squared
- Adjusted R-squared (penalises for extra variables)
- Residual Standard Error (how far off predictions typically are, in units of bikes)

In [ ]:
def model_comparison_table(models, names):
    """
    Create a model comparison table — like huxtable::huxreg() in R.
    
    Shows key fit statistics for multiple models side by side,
    making it easy to see whether adding more predictors improves the model.
    
    Arguments:
        models -- list of fitted statsmodels OLS objects
        names  -- list of model labels (strings)
    
    Returns:
        A nicely formatted pandas DataFrame
    """
    rows = []
    for model, name in zip(models, names):
        rows.append({
            'Model': name,
            'Formula': model.model.formula,
            'N (observations)': int(model.nobs),
            'R²': round(model.rsquared, 4),
            'Adj. R²': round(model.rsquared_adj, 4),
            'Residual SE': round(np.sqrt(model.mse_resid), 1),  # like sigma in R
            'AIC': round(model.aic, 1),   # lower AIC = better model fit (penalises complexity)
            'BIC': round(model.bic, 1)    # similar to AIC but with stronger penalty for complexity
        })
    return pd.DataFrame(rows).set_index('Model')

# Compare all models side by side
comparison = model_comparison_table(
    models=[model0, model1, model2, model3, model4, model5, model6],
    names=['Model 0 (mean)', 'Model 1 (tempmin)', 'Model 2 (feelslike+wknd+vis)',
           'Model 3 (+wday)', 'Model 4 (+humidity+month)', 
           'Model 5 (+precip+wind)', 'Model 6 (+cloud+moon)']
)

print("=== Model Comparison Table ===")
comparison

In [ ]:
# Visualise R² progression across models
# This shows how much explanatory power each model adds.

fig, ax = plt.subplots(figsize=(10, 5))

model_labels = comparison.index.tolist()
r2_values    = comparison['R²'].tolist()
adj_r2_values = comparison['Adj. R²'].tolist()

x = range(len(model_labels))

# Plot both R² and Adj. R² so we can see if adding variables truly helps
ax.plot(x, r2_values,     marker='o', label='R²',      linewidth=2)
ax.plot(x, adj_r2_values, marker='s', label='Adj. R²', linewidth=2, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels([f'M{i}' for i in range(len(model_labels))], fontsize=10)
ax.set_xlabel('Model')
ax.set_ylabel('R²')
ax.set_title('R² vs. Adjusted R² Across Models', fontsize=13)
ax.legend()
ax.set_ylim(0, 1)  # R² is always between 0 and 1

# Add a small annotation showing R² values
for xi, (r2, adj) in enumerate(zip(r2_values, adj_r2_values)):
    ax.annotate(f"{r2:.3f}", (xi, r2), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='steelblue')

plt.tight_layout()
plt.show()

---
## 11. Further Exploration

Our dataset has many more variables to explore! Here are some ideas:

1. **Are other weather variables useful?** Try adding `solarradiation`, `uvindex`, or `snowdepth`.
2. **Interactions**: Does the effect of temperature depend on the season?
   Try `bikes_hired ~ temp * season_name` (interaction term).
3. **What's your best model?** Use the comparison table to track improvements.
4. **Prediction vs. Explanation**: If we use this model to *predict*, the Residual SE
   (in units of bikes/day) tells us our typical prediction error.

Try adding your own model below! 👇

In [ ]:
# --- YOUR MODEL HERE ---
# Modify the formula below to try your own combination of predictors!
# Tip: use C(variable) for categorical variables (strings or factors)

my_model = fit_and_plot(
    formula='bikes_hired ~ feelslikemax + C(wday) + humidity + C(month) + precip + cloudcover',
    data=bike,
    model_name='My Custom Model'
)

print(f"\nMy model R²:      {my_model.rsquared:.4f}")
print(f"My model Adj. R²: {my_model.rsquared_adj:.4f}")
print(f"My model RSE:     {np.sqrt(my_model.mse_resid):.1f} bikes/day")